# Dataset Viewer

Drag the slider to flip through any dataset, one image at a time.

Works with MNIST, FashionMNIST, CIFAR-10 — or any torchvision-style dataset where indexing returns an `(image, label)` pair.

## Step 1 — Load a dataset

Uncomment exactly one of the lines below. The rest of the notebook only needs a variable called `dataset`.

In [ ]:
from torchvision import datasets, transforms

transform = transforms.ToTensor()

# --- Pick ONE: ---
dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
# dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
# dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)

print(f'Loaded {len(dataset)} images')
print(f'Classes: {dataset.classes}')

## Step 2 — Browse the images

Run the cell below. A control panel will appear with:

- **⏮ First** and **Last ⏭** — jump to the very first / very last image
- **← Previous** and **Next →** — step one image at a time
- **Random** — pick a random image (great for skimming the dataset)
- **Go to:** — type any image number and hit Enter to jump straight to it

A counter shows how many images are in the dataset and which one you're looking at.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

total = len(dataset)
current_idx = 0

# --- Build the controls ---
prev_btn   = widgets.Button(description='← Previous',  layout=widgets.Layout(width='120px'))
next_btn   = widgets.Button(description='Next →',      layout=widgets.Layout(width='120px'))
random_btn = widgets.Button(description='Random',      layout=widgets.Layout(width='100px'))
first_btn  = widgets.Button(description='⏮ First',     layout=widgets.Layout(width='100px'))
last_btn   = widgets.Button(description='Last ⏭',      layout=widgets.Layout(width='100px'))

idx_input  = widgets.BoundedIntText(value=0, min=0, max=total - 1,
                                    description='Go to:',
                                    layout=widgets.Layout(width='180px'))
counter    = widgets.HTML(value='')

picture    = widgets.Output()

def render(idx):
    global current_idx
    current_idx = idx

    # Sync the "Go to" box without re-triggering the observer
    if idx_input.value != idx:
        idx_input.unobserve(on_idx_change, names='value')
        idx_input.value = idx
        idx_input.observe(on_idx_change, names='value')

    counter.value = (
        f"<b>Image {idx + 1:,} of {total:,}</b> "
        f"<span style='color:#888'>(index {idx})</span>"
    )

    image, label = dataset[idx]
    if isinstance(image, torch.Tensor):
        image = image.numpy()
        if image.ndim == 3:
            image = image.transpose(1, 2, 0)
    image = np.squeeze(image)
    cmap = 'gray' if image.ndim == 2 else None
    class_name = dataset.classes[label] if hasattr(dataset, 'classes') else label

    with picture:
        clear_output(wait=True)
        plt.figure(figsize=(4, 4))
        plt.imshow(image, cmap=cmap)
        plt.title(f'Label: {label}  ({class_name})')
        plt.axis('off')
        plt.show()

# --- Wire up the buttons ---
def on_prev(b):   render((current_idx - 1) % total)
def on_next(b):   render((current_idx + 1) % total)
def on_random(b): render(int(np.random.randint(0, total)))
def on_first(b):  render(0)
def on_last(b):   render(total - 1)
def on_idx_change(change): render(change['new'])

prev_btn.on_click(on_prev)
next_btn.on_click(on_next)
random_btn.on_click(on_random)
first_btn.on_click(on_first)
last_btn.on_click(on_last)
idx_input.observe(on_idx_change, names='value')

# --- Lay it out and show ---
row1 = widgets.HBox([first_btn, prev_btn, next_btn, last_btn, random_btn])
row2 = widgets.HBox([idx_input, counter])
display(widgets.VBox([row1, row2, picture]))
render(0)

### Tips

- **Switch dataset:** go back to Step 1, comment out the current line, uncomment another, run it again, then re-run Step 2.
- **Your own dataset:** as long as `dataset[i]` returns `(image, label)` and `len(dataset)` works, the viewer will handle it — grayscale or colour, tensors or PIL images.